In [1]:
!pip install google-genai torch torchvision opencv-python-headless


In [2]:
# Install the CUDA 12 specific version of cuCIM
!pip install cucim-cu12

# Install CuPy for CUDA 12
!pip install cupy-cuda12x

# Install the required image format plugins
!pip install pylibcucim-cu12

  Using cached cucim_cu12-26.2.0.tar.gz (3.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [67 lines of output]
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl against tag cp310-cp310-manylinux_2_26_aarch64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl against tag cp310-cp310-manylinux_2_28_aarch64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl against tag cp310-cp310-manylinux_2_27_x86_64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl against tag cp310-cp310-manylinux_2_28_x86_64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp311-cp311-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl against tag cp311-cp311-manylinux_2_28_aarch64
      INFO:wheel-stub:Testing wheel cuci

In [3]:
from IPython.display import display, Image, Audio
import openai
import cv2  # We're using OpenCV to read video, to install !pip install opencv-python
import base64
import time
from openai import OpenAI
import os
import requests
import torch

In [4]:
# In a Python cell
!lspci | grep -i nvidia
!nvidia-smi
!nvcc --version
!which python
!python --version

'lspci' is not recognized as an internal or external command,
operable program or batch file.


Tue Apr 28 22:52:40 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 539.28                 Driver Version: 539.28       CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 NVL              TCC   | 00000000:01:00.0 Off |                    0 |
| N/A   30C    P0              61W / 400W |      1MiB / 95830MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

'which' is not recognized as an internal or external command,
operable program or batch file.


Python 3.10.11


In [5]:
!nvidia-smi

Tue Apr 28 22:52:41 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 539.28                 Driver Version: 539.28       CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 NVL              TCC   | 00000000:01:00.0 Off |                    0 |
| N/A   30C    P0              61W / 400W |      1MiB / 95830MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [6]:
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Opeyemi\PROMPTS\API-KEYS\multi-object-tracking-491921-9e3bbf535a36.json"
os.environ["GOOGLE_CLOUD_PROJECT"] = "multi-object-tracking-491921"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

print("✓ Google Cloud credentials configured")
print(f"  Project: {os.environ['GOOGLE_CLOUD_PROJECT']}")
print(f"  Credentials: {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")
print(f"  Location: {os.environ['GOOGLE_CLOUD_LOCATION']}")


✓ Google Cloud credentials configured
  Project: multi-object-tracking-491921
  Credentials: C:\Opeyemi\PROMPTS\API-KEYS\multi-object-tracking-491921-9e3bbf535a36.json
  Location: us-central1


## Dataset: UCF-Crime — Pre-Extracted Frames

**Frames must be pre-extracted before running any technique.**

Expected folder structure (either layout supported):
```
# Sub-folder layout (recommended):
FRAMES_DIR/
  Abuse/
    Abuse001_x264/   frame_000000.png  frame_000030.png  ...
    Abuse002_x264/   ...
  Arrest/ Arson/ Assault/ Burglary/ Explosion/ Fighting/
  RoadAccidents/ Robbery/ Shooting/ Shoplifting/ Stealing/ Vandalism/
```

**Configuration** (top of each technique cell):
```python
DATA_DIR   = r"C:\Opeyemi\PROMPTS\UCF-Data"   # original videos (reference only)

FRAMES_DIR = r"C:\Opeyemi\PROMPTS\FRAMES"      # pre-extracted frames

FRAME_INTERVAL = 1   # 1 = all frames | 2 = every 2nd frame | etc.
```


# Shared Utilities: Checkpoint System + NeurIPS Compute Tracker
Run this cell before any technique cell. It provides:
- **CheckpointManager** — saves per-chunk results to disk; resumes after network failures
- **ComputeTracker** — accumulates token estimates, API call stats, timing for NeurIPS reporting

In [7]:
# =============================================================================
# SHARED UTILITIES: Checkpoint System + NeurIPS Compute Tracker
# Run this cell BEFORE any technique cell.
# =============================================================================
import os, json, time


class CheckpointManager:
    """
    Persists per-chunk API responses to disk so that a network failure or
    kernel restart never causes already-finished work to be repeated.

    Layout:
      <save_dir>/_checkpoints/<Technique>_<video_key>.json
      {
        "complete": false,
        "chunks":   {"chunk_key": "<response_text>", ...},
        "result":   null          # filled when the video is fully processed
      }
    """
    def __init__(self, save_dir, technique_name):
        self.cp_dir    = os.path.join(save_dir, "_checkpoints")
        self.technique = technique_name
        os.makedirs(self.cp_dir, exist_ok=True)

    def _path(self, vk):
        safe = vk.replace("/", "_").replace(" ", "_")[:120]
        return os.path.join(self.cp_dir, f"{self.technique}_{safe}.json")

    def _read(self, vk):
        p = self._path(vk)
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)
        return {"complete": False, "chunks": {}, "result": None}

    def _write(self, vk, state):
        with open(self._path(vk), "w") as f:
            json.dump(state, f, indent=2)

    # ── video-level ───────────────────────────────────────────────────────────
    def is_video_complete(self, vk):
        return self._read(vk).get("complete", False)

    def get_video_result(self, vk):
        return self._read(vk).get("result")

    def mark_video_complete(self, vk, result):
        s = self._read(vk)
        s.update({"complete": True, "result": result,
                  "done_at": time.strftime("%Y%m%d_%H%M%S")})
        self._write(vk, s)

    # ── chunk-level ───────────────────────────────────────────────────────────
    def is_chunk_done(self, vk, ck):
        return ck in self._read(vk).get("chunks", {})

    def get_chunk(self, vk, ck):
        return self._read(vk).get("chunks", {}).get(ck)

    def save_chunk(self, vk, ck, text):
        s = self._read(vk)
        s.setdefault("chunks", {})[ck] = text
        s["last_updated"] = time.strftime("%Y%m%d_%H%M%S")
        self._write(vk, s)


class ComputeTracker:
    """
    Accumulates per-call statistics for NeurIPS-style compute reporting.

    Token estimation methodology
    ─────────────────────────────
    Text input  : characters / 4  (standard English approximation)
    Image input : 258 tokens / image  (Gemini vision billing constant)
    Output      : usageMetadata.candidatesTokenCount when present,
                  otherwise output_chars / 4
    """
    _IMG_TOK = 258

    def __init__(self, model_name, technique_name):
        self.model     = model_name
        self.technique = technique_name
        self._t0       = time.time()
        self.calls = self.ok = self.failed = 0
        self.in_tok = self.out_tok = self.img_count = 0
        self.latencies = []
        self.temps     = set()
        self.videos = self.frames = 0

    def record(self, *, success, prompt_chars, n_images, out_tok, temp, latency):
        self.calls += 1
        if success:
            self.ok += 1
        else:
            self.failed += 1
        self.in_tok    += max(prompt_chars, 0) // 4
        self.img_count += n_images
        self.out_tok   += out_tok
        self.latencies.append(latency)
        self.temps.add(round(float(temp), 2))

    def record_video(self, n_frames):
        self.videos += 1
        self.frames += n_frames

    def report(self):
        elapsed  = time.time() - self._t0
        avg_lat  = sum(self.latencies) / max(len(self.latencies), 1)
        img_tok  = self.img_count * self._IMG_TOK
        total    = self.in_tok + img_tok + self.out_tok
        return {
            "NeurIPS_Compute_Report": {
                "model"      : self.model,
                "technique"  : self.technique,
                "wall_clock" : {
                    "seconds": round(elapsed, 2),
                    "hours"  : round(elapsed / 3600, 5)
                },
                "api_calls"  : {
                    "total"         : self.calls,
                    "successful"    : self.ok,
                    "failed"        : self.failed,
                    "avg_latency_s" : round(avg_lat, 3)
                },
                "token_budget": {
                    "text_input_est"  : self.in_tok,
                    "image_input_est" : img_tok,
                    "output"          : self.out_tok,
                    "grand_total_est" : total,
                    "images_sent"     : self.img_count,
                    "methodology"     : (
                        "text_input: chars/4 | "
                        "image_input: 258 tok/image (Gemini vision pricing) | "
                        "output: usageMetadata.candidatesTokenCount when available"
                    )
                },
                "data_processed"  : {"videos": self.videos, "frames": self.frames},
                "hyperparameters" : {
                    "temperatures"      : sorted(self.temps),
                    "max_output_tokens" : 4096,
                    "top_p"             : 0.8,
                    "top_k"             : 10,
                    "chunk_size"        : 10
                },
                "reproducibility": {
                    "model_string": self.model,
                    "api_endpoint": "Vertex AI (google-genai SDK)",
                    "api_version" : "vertex-ai-genai",
                    "random_seed" : (
                        "N/A - Gemini REST API does not expose a seed parameter. "
                        "At temperature=0.1, outputs are near-deterministic; "
                        "minor token-sampling variation is possible across runs."
                    )
                }
            }
        }


print("Checkpoint and compute-tracking utilities loaded.")


Checkpoint and compute-tracking utilities loaded.


#Iterative prompting
Iterative Prompting Approach
The Iterative Prompting technique follows a structured refinement process:

- Initial Analysis: The model provides a first-pass analysis of what appears to be happening in the frames
- Guided Iterations: Through a series of targeted follow-up prompts, the model refines specific aspects of its analysis
- Progressive Improvement: Each round builds on previous insights while addressing potential weaknesses or gaps
- Final Synthesis: After multiple refinement rounds, the model creates a final, comprehensive assessment

Implementation Highlights

Multi-Round Refinement:

Starts with an initial general analysis prompt
Follows with 4 specialized refinement rounds:

- People and relationships focus
- Actions and intent focus
- Criminal elements and evidence focus
- Critical examination (missing elements, alternative interpretations)


Concludes with a final synthesis prompt for each chunk


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full iterative process independently


Conversation Continuity:

- Maintains the complete conversation history throughout all rounds
- Each refinement builds on the accumulated context from previous rounds
- Creates a progressive improvement cycle where later responses incorporate earlier insights


Holistic Synthesis:

- After all chunks are iteratively analyzed, performs a final cross-chunk synthesis
- Creates a coherent narrative of the entire incident
- Addresses any discrepancies between chunk analyses

In [ ]:
import os
import json
import re  
import base64
import requests
from google import genai
from google.genai import types
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# Running on local machine
# Ensure Google Cloud credentials are set (run cell 5 first)
if "GOOGLE_APPLICATION_CREDENTIALS" not in os.environ:
    print("⚠ WARNING: Run the credentials cell (cell 5) first!")
    print("  GOOGLE_APPLICATION_CREDENTIALS not set")

# Configuration
DATA_DIR   = "C:\\Opeyemi\\PROMPTS\\UCF-Data"   # root with crime-type subfolders
FRAMES_DIR = r"C:\\Opeyemi\\PROMPTS\\FRAMES"  # pre-extracted frames
SAVE_DIR = "C:\\Opeyemi\\PROMPTS\\RESULTS\\GEMINI\\ITERATIVE"
FRAME_INTERVAL = 1  # Sample every Nth frame (1 = all frames)
MAX_WORKERS    = 4    # parallel videos processed at once (Gemini/Vertex quotas; raise carefully)
BATCH_SIZE     = 20   # frames per API call (must be defined per-cell so cells run independently)

class TrueIterativeGeminiAnalyzer:
    def __init__(self):
        self.model_name = "gemini-3.1-pro-preview"
        self.client = genai.Client(
            vertexai=True,
            project=os.environ["GOOGLE_CLOUD_PROJECT"],
            location="global",
        )
        self.save_dir = SAVE_DIR
        self.max_frames_per_request = 20
        self.chunk_size = BATCH_SIZE
        self.max_iterations = 8  # Maximum iterations before stopping
        self.convergence_threshold = 0.7  # Similarity threshold for convergence
        os.makedirs(self.save_dir, exist_ok=True)
        self._ckpt = CheckpointManager(self.save_dir, type(self).__name__)
        self._comp = ComputeTracker(self.model_name, type(self).__name__)
        # Core iterative question that gets refined each time
        self.core_question = "Analyze these video frames for criminal activity. What crime is occurring, who is involved, what evidence supports your conclusion, and how confident are you in this assessment?"


    def _call_gemini_sdk_from_payload(self, payload):
        """Convert REST API payload to genai SDK call and return REST-compatible response dict."""
        try:
            # Extract parts from payload
            contents_list = payload.get("contents", [])
            gen_config = payload.get("generationConfig", {})
            
            # Build SDK contents
            sdk_parts = []
            for content_block in contents_list:
                for part in content_block.get("parts", []):
                    if "text" in part:
                        sdk_parts.append(types.Part.from_text(text=part["text"]))
                    elif "inline_data" in part:
                        mime = part["inline_data"].get("mime_type", "image/png")
                        data_bytes = base64.b64decode(part["inline_data"]["data"])
                        sdk_parts.append(types.Part.from_bytes(data=data_bytes, mime_type=mime))
            
            sdk_content = types.Content(role="user", parts=sdk_parts)
            
            # Build generation config
            config = types.GenerateContentConfig(
                temperature=gen_config.get("temperature", 0.1),
                max_output_tokens=gen_config.get("maxOutputTokens", 4096),
                top_p=gen_config.get("topP", 0.8),
                top_k=gen_config.get("topK", 10),
            )
            
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=sdk_content,
                config=config,
            )
            
            # Convert to REST-compatible dict
            if response.text:
                return {
                    "candidates": [{
                        "content": {
                            "parts": [{"text": response.text}]
                        }
                    }],
                    "usageMetadata": {
                        "promptTokenCount": getattr(response.usage_metadata, 'prompt_token_count', 0) if response.usage_metadata else 0,
                        "candidatesTokenCount": getattr(response.usage_metadata, 'candidates_token_count', 0) if response.usage_metadata else 0,
                    }
                }
            else:
                return {"error": "No text in response", "candidates": []}
                
        except Exception as e:
            error_msg = str(e)
            print(f"API Error: {error_msg}")
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "quota" in error_msg.lower():
                print("Rate limited - waiting 60 seconds...")
                time.sleep(60)
                return self._call_gemini_sdk_from_payload(payload)  # Retry once
            return {"error": error_msg}


    def make_gemini_request(self, contents):
        """Make a single request to Gemini API"""
        payload = {
            "contents": contents,
            "generationConfig": {
                "temperature": 0.1,
                "maxOutputTokens": 4096,
                "topP": 0.8,
                "topK": 10
            }
        }
        _t0_req = time.time()
        result = self._call_gemini_sdk_from_payload(payload)
        _latency = time.time() - _t0_req
        _prompt_chars = len(str(payload))
        _n_imgs = sum(1 for c in contents for p in c.get("parts", []) if "inline_data" in p)
        
        if isinstance(result, dict) and "error" in result:
            print(f"API Error: {result['error']}")
            self._comp.record(success=False, prompt_chars=_prompt_chars,
                              n_images=_n_imgs, out_tok=0,
                              temp=0.1, latency=_latency)
            return f"Error: {result['error']}"
        
        if "candidates" in result and result["candidates"]:
            candidate = result["candidates"][0]
            if "content" in candidate and "parts" in candidate["content"]:
                text = candidate["content"]["parts"][0]["text"]
                _usage = result.get("usageMetadata", {})
                _out_tok = _usage.get("candidatesTokenCount", len(text)//4)
                self._comp.record(success=True, prompt_chars=_prompt_chars,
                                  n_images=_n_imgs, out_tok=_out_tok,
                                  temp=0.1, latency=_latency)
                return text
        
        return "Error: No valid response from Gemini"

    def calculate_similarity(self, text1, text2):
        """Simple similarity calculation based on word overlap"""
        if not text1 or not text2:
            return 0.0

        # Convert to lowercase and split into words
        words1 = set(text1.lower().split())
        words2 = set(text2.lower().split())

        # Calculate Jaccard similarity
        intersection = words1.intersection(words2)
        union = words1.union(words2)

        if len(union) == 0:
            return 0.0

        return len(intersection) / len(union)

    def has_converged(self, current_response, previous_response):
        """Check if the analysis has converged (responses are very similar)"""
        if not previous_response:
            return False

        similarity = self.calculate_similarity(current_response, previous_response)
        print(f"    Similarity to previous: {similarity:.3f} (threshold: {self.convergence_threshold})")

        return similarity >= self.convergence_threshold

    def extract_confidence_score(self, response):
        """Extract confidence score from response if mentioned"""
        confidence_keywords = ["confidence", "confident", "certainty", "sure", "probability"]
        response_lower = response.lower()

        # Look for percentage mentions
        import re
        percentages = re.findall(r'(\d+)%', response)
        if percentages:
            return max([int(p) for p in percentages]) / 100.0

        # Look for confidence keywords with qualifiers
        if any(keyword in response_lower for keyword in ["very confident", "highly confident", "extremely confident"]):
            return 0.9
        elif any(keyword in response_lower for keyword in ["confident", "fairly confident"]):
            return 0.7
        elif any(keyword in response_lower for keyword in ["somewhat confident", "moderately confident"]):
            return 0.5
        elif any(keyword in response_lower for keyword in ["low confidence", "uncertain", "unsure"]):
            return 0.3

        return 0.5  # Default moderate confidence

    def process_frames_truly_iteratively(self, frame_data, video_id, crime_type):
        """Process frames using TRUE iterative prompting - same question refined repeatedly"""
        all_iterations = {}
        previous_response = None
        converged = False

        print(f"Starting TRUE iterative analysis with max {self.max_iterations} iterations...")
        print(f"Core question: {self.core_question}")
        print(f"Convergence threshold: {self.convergence_threshold}")

        # TRUE ITERATIVE LOOP - Same question, progressively refined
        _vk_iter = getattr(self, "_vk", "unknown_video")
        for iteration_num in range(1, self.max_iterations + 1):
            print(f"\n=== TRUE ITERATION {iteration_num}/{self.max_iterations} ===")

            iteration_responses = []

            # Process frames in chunks for this iteration
            for i in range(0, len(frame_data), self.chunk_size):
                chunk = frame_data[i:i + self.chunk_size]

                # Build TRUE iterative prompt
                if iteration_num == 1:
                    # First iteration - ask the core question
                    iterative_prompt = f"""ITERATION {iteration_num} - Initial Analysis

{self.core_question}

Be thorough and specific in your analysis. Include your confidence level in your assessment."""

                else:
                    # Subsequent iterations - refine based on previous response
                    iterative_prompt = f"""ITERATION {iteration_num} - Refining Previous Analysis

PREVIOUS ANALYSIS FROM ITERATION {iteration_num-1}:
{previous_response[:800]}...

Now, analyze these SAME frames again with the SAME core question, but refine your analysis:

{self.core_question}

REFINEMENT INSTRUCTIONS:
- Review your previous analysis carefully
- Look for details you may have missed
- Reconsider your conclusions with fresh perspective
- Identify any errors or oversights in your previous assessment
- Improve the accuracy and depth of your analysis
- If you're more confident now, explain why
- If you're less confident, explain what creates uncertainty
- What new insights do you have upon re-examination?

Provide your REFINED analysis of the same core question."""

                # Prepare content parts for Gemini
                parts = [{"text": iterative_prompt}]

                # Add images to parts
                for frame in chunk:
                    parts.append({
                        "inline_data": {
                            "mime_type": "image/png",
                            "data": frame
                        }
                    })

                contents = [{
                    "role": "user",
                    "parts": parts
                }]

                _ck_iter = f"iter_{iteration_num}_chunk_{i//self.chunk_size}"
                if self._ckpt.is_chunk_done(_vk_iter, _ck_iter):
                    print(f"  [CHECKPOINT] Iter {iteration_num} chunk {i//self.chunk_size+1} loaded from disk")
                    iteration_responses.append(self._ckpt.get_chunk(_vk_iter, _ck_iter))
                    continue
                print(f"  Processing chunk {i//self.chunk_size + 1}/{(len(frame_data) + self.chunk_size - 1)//self.chunk_size}...")

                # Make API request for this chunk
                response = self.make_gemini_request(contents)
                iteration_responses.append(response)
                if not response.startswith("Error"):
                    self._ckpt.save_chunk(_vk_iter, _ck_iter, response)

                # Rate limiting
                print(f"  Waiting 3 seconds before next request...")
                time.sleep(3)

            # Combine responses for this iteration
            if len(iteration_responses) == 1:
                current_response = iteration_responses[0]
            else:
                current_response = "\n\n=== NEXT CHUNK ===\n\n".join(iteration_responses)

            # Extract confidence for this iteration
            confidence = self.extract_confidence_score(current_response)

            # Check for convergence
            if previous_response:
                converged = self.has_converged(current_response, previous_response)

            # Store this iteration's data
            iteration_data = {
                "iteration": iteration_num,
                "type": "true_iterative_refinement",
                "core_question": self.core_question,
                "prompt_used": iterative_prompt,
                "response": current_response,
                "confidence_extracted": confidence,
                "converged": converged,
                "similarity_to_previous": self.calculate_similarity(current_response, previous_response) if previous_response else 0.0
            }

            all_iterations[f"iteration_{iteration_num}"] = iteration_data

            # Save intermediate results after each iteration
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            iteration_result = {
                f"iteration_{iteration_num}": iteration_data
            }
            self.save_results(iteration_result, f"{crime_type}_{video_id}_iterative_iteration{iteration_num}_{timestamp}.json")

            print(f"  Confidence level: {confidence:.2f}")
            print(f"  Response preview: {current_response[:200]}...")

            # Check for convergence
            if converged:
                print(f"  *** CONVERGENCE ACHIEVED at iteration {iteration_num} ***")
                break
            elif iteration_num < self.max_iterations:
                print(f"  Continuing to iteration {iteration_num + 1} (not yet converged)")

            # Update previous response for next iteration
            previous_response = current_response

        # Create convergence summary
        convergence_summary = {
            "total_iterations_run": iteration_num,
            "max_iterations_allowed": self.max_iterations,
            "converged": converged,
            "convergence_threshold": self.convergence_threshold,
            "final_confidence": confidence,
            "methodology": "True iterative refinement - same question refined repeatedly"
        }

        if converged:
            convergence_summary["convergence_iteration"] = iteration_num
            convergence_summary["convergence_reason"] = f"Response similarity reached {self.convergence_threshold} threshold"
        else:
            convergence_summary["convergence_reason"] = f"Maximum iterations ({self.max_iterations}) reached without convergence"

        return all_iterations, convergence_summary

    def save_results(self, results, filename):
        """Save results to a file"""
        filepath = os.path.join(self.save_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Results saved to: {filepath}")

    def analyze_frames(self, frames_data, video_id, crime_type):
        """Analyze frames with TRUE iterative prompting approach"""
        try:
            frame_names = list(frames_data.keys())

            # Improved sorting function for frame numbers
            def extract_frame_number(filename):
                try:
                    # Handle different naming patterns
                    if '_frame_' in filename:
                        parts = filename.split('_frame_')
                        if len(parts) > 1:
                            number_part = parts[1].split('.')[0]
                            return int(number_part)
                    elif 'frame' in filename.lower():
                        # Alternative pattern matching
                        import re
                        numbers = re.findall(r'\d+', filename)
                        if numbers:
                            return int(numbers[-1])  # Use the last number found
                except Exception as e:
                    print(f"Error extracting frame number from {filename}: {str(e)}")
                    return 0

            sorted_frames = sorted(frame_names, key=extract_frame_number)

            print(f"\n=== ANALYZING VIDEO: {video_id} ({crime_type}) ===")
            print(f"Total frames loaded: {len(frames_data)}")

            _vk = f"{crime_type}_{video_id}"
            if self._ckpt.is_video_complete(_vk):
                print(f"  [CHECKPOINT] {video_id} already complete, loading from disk")
                return self._ckpt.get_video_result(_vk)

            print(f"Frame names sample: {sorted_frames[:5]}{'...' if len(sorted_frames) > 5 else ''}")
            print(f"Using model: {self.model_name}")

            results = {}
            timestamp = time.strftime("%Y%m%d_%H%M%S")

            try:
                frame_data = [frames_data[frame_name] for frame_name in sorted_frames
                             if frame_name in frames_data and frames_data[frame_name]]

                if not frame_data:
                    results["True_Iterative_Analysis"] = {
                        "error": "No valid frames were available for analysis.",
                        "frames_used": len(sorted_frames),
                        "valid_frames": 0,
                        "model_used": self.model_name,
                        "crime_type": crime_type,
                        "prompting_technique": "TRUE_ITERATIVE_PROMPTING"
                    }
                    print("WARNING: No valid frames were available for analysis.")
                else:
                    print(f"Processing {len(frame_data)} valid frames with TRUE iterative prompting...")
                    iterative_responses, convergence_summary = self.process_frames_truly_iteratively(frame_data, video_id, crime_type)

                    results["True_Iterative_Analysis"] = {
                        "method": "true_iterative_prompting",
                        "description": "Same question refined repeatedly until convergence",
                        "core_question": self.core_question,
                        "convergence_summary": convergence_summary,
                        "all_iterations": iterative_responses,
                        "frames_used": len(sorted_frames),
                        "valid_frames": len(frame_data),
                        "analysis_timestamp": timestamp,
                        "model_used": self.model_name,
                        "crime_type": crime_type,
                        "prompting_technique": "TRUE_ITERATIVE_PROMPTING"
                    }

                # Save results
                self.save_results(results, f"{crime_type}_{video_id}_true_iterative_analysis_{timestamp}.json")
                self._ckpt.mark_video_complete(_vk, results)
                self._comp.record_video(len(frame_data))
                print(f"True iterative analysis for {video_id} ({crime_type}) completed and saved.")

            except Exception as e:
                print(f"Error processing true iterative analysis: {str(e)}")
                results["True_Iterative_Analysis"] = {
                    "method": "true_iterative_prompting",
                    "error": str(e),
                    "frames_used": len(sorted_frames) if 'sorted_frames' in locals() else 0,
                    "model_used": self.model_name,
                    "crime_type": crime_type,
                    "prompting_technique": "TRUE_ITERATIVE_PROMPTING"
                }

            return results

        except Exception as e:
            print(f"Error in analyze_frames: {str(e)}")
            raise

def discover_all_videos_and_frames(frames_dir):
    """
    Discover pre-extracted frames from FRAMES_DIR.
    Expected layout:
      FRAMES_DIR/<CrimeType>/<VideoID>/frame_000000.png ...
    or (flat):
      FRAMES_DIR/<CrimeType>/<VideoID_frame_XXXXXX>.png ...
    Both layouts are supported automatically.
    """
    print(f"\n=== DISCOVERING PRE-EXTRACTED FRAMES ===")
    print(f"Scanning: {frames_dir}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
    all_videos = {}

    try:
        crime_types = sorted([
            d for d in os.listdir(frames_dir)
            if os.path.isdir(os.path.join(frames_dir, d))
        ])
        print(f"Crime-type folders ({len(crime_types)}): {crime_types}")

        for crime_type in crime_types:
            crime_dir = os.path.join(frames_dir, crime_type)

            # Check for sub-folder layout: <CrimeType>/<VideoID>/<frames>
            sub_dirs = [
                d for d in os.listdir(crime_dir)
                if os.path.isdir(os.path.join(crime_dir, d))
            ]

            if sub_dirs:
                # Sub-folder layout
                for video_id in sorted(sub_dirs):
                    video_frame_dir = os.path.join(crime_dir, video_id)
                    frames = sorted([
                        f for f in os.listdir(video_frame_dir)
                        if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                    ])
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : frames,
                            "crime_dir"  : video_frame_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")
            else:
                # Flat layout: group image files by video_id prefix
                all_files = sorted([
                    f for f in os.listdir(crime_dir)
                    if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                ])
                video_groups = defaultdict(list)
                for fname in all_files:
                    # Extract video_id: everything before _frame_ or last _number
                    base = os.path.splitext(fname)[0]
                    if "_frame_" in base:
                        vid_id = base.split("_frame_")[0]
                    else:
                        vid_id = re.sub(r"_?\d+$", "", base) or base
                    video_groups[vid_id].append(fname)

                for video_id, frames in sorted(video_groups.items()):
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : sorted(frames),
                            "crime_dir"  : crime_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")

    except Exception as e:
        print(f"Error scanning {frames_dir}: {e}")

    print(f"\nTotal videos found: {len(all_videos)}")
    return all_videos
def extract_video_id_from_filename(filename):
    """Legacy stub — video IDs come from folder/file names directly."""
    base = os.path.splitext(filename)[0]
    if "_frame_" in base:
        return base.split("_frame_")[0]
    return re.sub(r"_?\d+$", "", base) or base
def load_frames_for_video(video_info, frame_interval=1):
    """
    Load pre-extracted frame images from disk and base64-encode them.
    Reads from video_info["crime_dir"] which points to the frame folder.
    frame_interval: sample every Nth frame (1 = all frames).
    """
    crime_dir  = video_info["crime_dir"]
    frame_files = video_info["frames"]
    video_id   = video_info["video_id"]

    print(f"\nLoading frames for {video_id} from: {crime_dir}")
    print(f"  Total available: {len(frame_files)} | sampling every {frame_interval}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

    # Sort by embedded frame number
    def _frame_num(fname):
        nums = re.findall(r"\d+", fname)
        return int(nums[-1]) if nums else 0

    sorted_files = sorted(frame_files, key=_frame_num)
    selected     = sorted_files[::frame_interval]
    print(f"  Frames to load: {len(selected)}")

    frames_data = {}
    for idx, fname in enumerate(selected):
        if os.path.splitext(fname.lower())[1] not in IMAGE_EXTS:
            continue
        fpath = os.path.join(crime_dir, fname)
        try:
            with open(fpath, "rb") as fh:
                frames_data[fname] = base64.b64encode(fh.read()).decode("utf-8")
            if idx < 3 or idx % 20 == 0 or idx == len(selected) - 1:
                print(f"  [{idx+1:>5}] {fname} ({os.path.getsize(fpath)/1024:.1f} KB)")
        except Exception as e:
            print(f"  Error loading {fname}: {e}")

    print(f"  Done: {len(frames_data)} frames loaded")
    return frames_data
def process_all_crime_folders():
    """Process ALL crime folders with true iterative analysis - PROCESSES ENTIRE DIRECTORY STRUCTURE"""
    # Initialize analyzer
    analyzer = TrueIterativeGeminiAnalyzer()

    # Discover ALL videos and frames in the entire folder structure
    all_videos = discover_all_videos_and_frames(FRAMES_DIR)

    if not all_videos:
        print("No videos found to process!")
        return {}

    all_results = {}
    skipped_videos = []

    print(f"\n🔥 PROCESSING ALL {len(all_videos)} VIDEOS WITH TRUE ITERATIVE PROMPTING 🔥")
    print(f"Using model: {analyzer.model_name}")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("🎯 True Iterative Mode: Same question refined repeatedly until convergence!")
    print("📁 ENTIRE FOLDER STRUCTURE WILL BE PROCESSED")
    print("="*70)

    # Process EVERY SINGLE VIDEO discovered
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(video_key, video_info):
        """Process a single video in its own thread.

        Preserves the original per-video checkpoint (analyzer._ckpt) so
        already-completed videos are skipped on resume. Checkpoint reads
        and result-dict writes are protected by checkpoint_lock; stdout
        writes are protected by print_lock.
        """
        with print_lock:
            print(f"\nProcessing video: {video_key}")
            print(f"  Crime type: {video_info['crime_type']}")
            print(f"  Video ID: {video_info['video_id']}")
            print(f"  Frames available: {len(video_info['frames'])}")
        try:
            frames_data = load_frames_for_video(video_info, frame_interval=FRAME_INTERVAL)
            if not frames_data:
                with print_lock:
                    print(f"  No frames loaded for video {video_key} - skipping")
                return video_key, None, "no frames loaded"

            # Resume: skip videos already finished in a prior run
            if analyzer._ckpt.is_video_complete(video_key):
                with print_lock:
                    print(f"  [CHECKPOINT] {video_key} already complete, loading from disk")
                cached = analyzer._ckpt.get_video_result(video_key)
                with checkpoint_lock:
                    all_results[video_key] = cached
                    analyzer._comp.record_video(len(frames_data))
                return video_key, cached, None

            # Run the technique
            results = analyzer.analyze_frames(
                frames_data, video_info['video_id'], video_info['crime_type']
            )
            with checkpoint_lock:
                all_results[video_key] = results
            with print_lock:
                print(f"  Successfully processed {video_key}")
            return video_key, results, None

        except Exception as e:
            with print_lock:
                print(f"  Error processing video {video_key}: {e}")
            return video_key, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(_process_one_video, k, v)
            for k, v in all_videos.items()
        ]
        for fut in as_completed(futures):
            vkey, _res, err = fut.result()
            if err:
                skipped_videos.append(f"{vkey} ({err})")

    # Save summary results
    summary_file = os.path.join(SAVE_DIR, f"true_iterative_summary_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(summary_file, 'w') as f:
        json.dump(all_results, f, indent=2)

    # Log skipped videos
    if skipped_videos:
        skipped_file = os.path.join(SAVE_DIR, f"skipped_videos_{time.strftime('%Y%m%d_%H%M%S')}.txt")
        with open(skipped_file, 'w') as f:
            f.write("Videos that could not be processed:\n")
            for video in skipped_videos:
                f.write(f"{video}\n")
        print(f"\nSkipped {len(skipped_videos)} videos. List saved to: {skipped_file}")

    print(f"\nComplete TRUE iterative analysis saved to: {summary_file}")
    print(f"Successfully processed {len(all_results)} videos out of {len(all_videos)} total")

    return all_results

def test_gemini_api():
    """Test Gemini API connection via Vertex AI genai SDK"""
    print("\nTesting Gemini API connection via Vertex AI...")
    try:
        client = genai.Client(
            vertexai=True,
            project=os.environ["GOOGLE_CLOUD_PROJECT"],
            location="global",
        )
        response = client.models.generate_content(
            model="gemini-3.1-pro-preview",
            contents="Hello, respond with 'API connection successful'"
        )
        if response.text:
            print(f"✓ Gemini API connection successful!")
            print(f"Response: {response.text[:100]}")
            return True
        else:
            print("✗ No response text received")
            return False
    except Exception as e:
        print(f"✗ API connection failed: {e}")
        return False

def check_authentication():
    """Placeholder function to check authentication"""
    return True

def run():
    """Main execution function"""
    print("TRUE Iterative Prompting Crime Video Analysis with Gemini - ENTIRE FOLDER PROCESSING")
    print("="*70)
    print("🎯 TRUE ITERATIVE TECHNIQUE: Same question refined repeatedly until convergence!")
    print("📁 PROCESSES ALL VIDEOS IN ALL CRIME TYPE FOLDERS")
    print("="*70)

    # Test directory access first
    print("Testing directory access...")
    for path in [DATA_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    # Vertex AI uses GOOGLE_CLOUD_PROJECT env var (no API key file needed)
    print(f"GOOGLE_CLOUD_PROJECT: {os.environ.get('GOOGLE_CLOUD_PROJECT', 'NOT SET')}")

    # Test Gemini API connection
    if not test_gemini_api():
        print("✗ Gemini API test failed. Please check your API key and connection.")
        return

    # Check authentication
    if not check_authentication():
        print("✗ Authentication not completed.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(DATA_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process ALL crime folders - ENTIRE DIRECTORY STRUCTURE
    print("\n🚀 STARTING COMPLETE FOLDER PROCESSING WITH TRUE ITERATIVE PROMPTING...")
    results = process_all_crime_folders()

    # NeurIPS compute report
    if 'analyzer' in dir():
        _report = analyzer._comp.report()
    else:
        _report = {"note": "analyzer not in scope"}
    _rpath = os.path.join(SAVE_DIR, f"neurips_compute_report_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(_rpath, 'w') as _rf:
        json.dump(_report, _rf, indent=2)
    print(f"NeurIPS compute report saved: {_rpath}")

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_iterations_run = 0
    convergence_achieved = 0

    for video_id, video_results in results.items():
        if video_results and 'True_Iterative_Analysis' in video_results:
            analysis = video_results['True_Iterative_Analysis']
            total_frames_processed += analysis.get('valid_frames', 0)
            if 'convergence_summary' in analysis:
                total_iterations_run += analysis['convergence_summary'].get('total_iterations_run', 0)
                if analysis['convergence_summary'].get('converged', False):
                    convergence_achieved += 1

    print("\n" + "="*70)
    print(f"🎉 COMPLETE TRUE ITERATIVE PROCESSING FINISHED!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total iterations run: {total_iterations_run}")
    print(f"Convergence achieved: {convergence_achieved}/{total_videos_processed} videos")
    print(f"Model used: {analyzer.model_name if 'analyzer' in locals() else 'gemini-3.1-pro-preview'}")
    print(f"Method: Same core question refined repeatedly until convergence")
    print("📁 ENTIRE FOLDER STRUCTURE WAS PROCESSED")
    print("🎯 True iterative prompting technique applied to all videos")
    print("="*70)

run()

TRUE Iterative Prompting Crime Video Analysis with Gemini - ENTIRE FOLDER PROCESSING
🎯 TRUE ITERATIVE TECHNIQUE: Same question refined repeatedly until convergence!
📁 PROCESSES ALL VIDEOS IN ALL CRIME TYPE FOLDERS
Testing directory access...
Path: C:\Opeyemi\PROMPTS\UCF-Data
  Exists: True
  Contains 28 items
  First few items: ['.DS_Store', '._.DS_Store', '._Abuse']
Path: C:\Opeyemi\PROMPTS\RESULTS\GEMINI\ITERATIVE
  Exists: False
GOOGLE_CLOUD_PROJECT: multi-object-tracking-491921

Testing Gemini API connection via Vertex AI...
✓ Gemini API connection successful!
Response: API connection successful

Verifying directories:
Data directory exists: True
Save directory exists: False

🚀 STARTING COMPLETE FOLDER PROCESSING WITH TRUE ITERATIVE PROMPTING...

=== DISCOVERING PRE-EXTRACTED FRAMES ===
Scanning: C:\\Opeyemi\\PROMPTS\\FRAMES
Crime-type folders (11): ['Abuse', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalis

#Meta-Prompting
- Meta-Prompting that processes all frames from crime videos. This technique is unique because it uses the AI to generate its own specialized prompts for analysis.

Meta-Prompting Approach
The Meta-Prompting technique follows this innovative process:

- Prompt Generation: Instead of using predefined prompts, the system asks the AI to create specialized prompts for analyzing video frames
- Prompt Application: These AI-generated prompts are then used to analyze the actual frames
- Meta-Synthesis: The system also generates a specialized synthesis prompt to combine all chunk analyses

Implementation Highlights

Two-Stage Meta-Prompting:

- First Stage: For each chunk of frames, generate a specialized analysis prompt
- Second Stage: For final synthesis, generate a specialized synthesis prompt
- Both stages use the AI to create task-specific prompts rather than using predefined ones


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full meta-prompting process independently


Specialized Prompt Design Process: Guides the AI to create prompts that focus on:

Step-by-step observation:
- Objective description before interpretation
- Attention to easily missed details
- Organizing observations into a coherent narrative
- Avoids including example responses in the generated prompts


Fallback Safety:

- If meta-prompting fails, falls back to a simple seed prompt
Ensures analysis can continue even if prompt generation has issues

In [ ]:
import os
import json
import base64
import re  
import requests
from google import genai
from google.genai import types
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# Running on local machine
# Ensure Google Cloud credentials are set (run cell 5 first)
if "GOOGLE_APPLICATION_CREDENTIALS" not in os.environ:
    print("⚠ WARNING: Run the credentials cell (cell 5) first!")
    print("  GOOGLE_APPLICATION_CREDENTIALS not set")

# Configuration
DATA_DIR   = "C:\\Opeyemi\\PROMPTS\\UCF-Data"   # root with crime-type subfolders
FRAMES_DIR = r"C:\\Opeyemi\\PROMPTS\\FRAMES"  # pre-extracted frames
SAVE_DIR = "C:\\Opeyemi\\PROMPTS\\RESULTS\\GEMINI\\META-PROMPTING"
FRAME_INTERVAL = 1  # Sample every Nth frame (1 = all frames)
MAX_WORKERS    = 4    # parallel videos processed at once (Gemini/Vertex quotas; raise carefully)
BATCH_SIZE     = 20   # frames per API call (must be defined per-cell so cells run independently)

class MetaPromptingGeminiAnalyzer:
    def __init__(self):
        self.model_name = "gemini-3.1-pro-preview"
        self.client = genai.Client(
            vertexai=True,
            project=os.environ["GOOGLE_CLOUD_PROJECT"],
            location="global",
        )
        self.save_dir = SAVE_DIR
        self.max_frames_per_request = 20
        self.chunk_size = BATCH_SIZE
        os.makedirs(self.save_dir, exist_ok=True)
        self._ckpt = CheckpointManager(self.save_dir, type(self).__name__)
        self._comp = ComputeTracker(self.model_name, type(self).__name__)


    def _call_gemini_sdk_from_payload(self, payload):
        """Convert REST API payload to genai SDK call and return REST-compatible response dict."""
        try:
            # Extract parts from payload
            contents_list = payload.get("contents", [])
            gen_config = payload.get("generationConfig", {})
            
            # Build SDK contents
            sdk_parts = []
            for content_block in contents_list:
                for part in content_block.get("parts", []):
                    if "text" in part:
                        sdk_parts.append(types.Part.from_text(text=part["text"]))
                    elif "inline_data" in part:
                        mime = part["inline_data"].get("mime_type", "image/png")
                        data_bytes = base64.b64decode(part["inline_data"]["data"])
                        sdk_parts.append(types.Part.from_bytes(data=data_bytes, mime_type=mime))
            
            sdk_content = types.Content(role="user", parts=sdk_parts)
            
            # Build generation config
            config = types.GenerateContentConfig(
                temperature=gen_config.get("temperature", 0.1),
                max_output_tokens=gen_config.get("maxOutputTokens", 4096),
                top_p=gen_config.get("topP", 0.8),
                top_k=gen_config.get("topK", 10),
            )
            
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=sdk_content,
                config=config,
            )
            
            # Convert to REST-compatible dict
            if response.text:
                return {
                    "candidates": [{
                        "content": {
                            "parts": [{"text": response.text}]
                        }
                    }],
                    "usageMetadata": {
                        "promptTokenCount": getattr(response.usage_metadata, 'prompt_token_count', 0) if response.usage_metadata else 0,
                        "candidatesTokenCount": getattr(response.usage_metadata, 'candidates_token_count', 0) if response.usage_metadata else 0,
                    }
                }
            else:
                return {"error": "No text in response", "candidates": []}
                
        except Exception as e:
            error_msg = str(e)
            print(f"API Error: {error_msg}")
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "quota" in error_msg.lower():
                print("Rate limited - waiting 60 seconds...")
                time.sleep(60)
                return self._call_gemini_sdk_from_payload(payload)  # Retry once
            return {"error": error_msg}

    def make_gemini_request(self, contents, temperature=0.1):
        """Make a single request to Gemini API"""
        payload = {
            "contents": contents,
            "generationConfig": {
                "temperature": 0.1,
                "maxOutputTokens": 4096,
                "topP": 0.8,
                "topK": 10
            }
        }
        _t0_req = time.time()
        result = self._call_gemini_sdk_from_payload(payload)
        _latency = time.time() - _t0_req
        _prompt_chars = len(str(payload))
        _n_imgs = sum(1 for c in contents for p in c.get("parts", []) if "inline_data" in p)
        
        if isinstance(result, dict) and "error" in result:
            print(f"API Error: {result['error']}")
            self._comp.record(success=False, prompt_chars=_prompt_chars,
                              n_images=_n_imgs, out_tok=0,
                              temp=0.1, latency=_latency)
            return f"Error: {result['error']}"
        
        if "candidates" in result and result["candidates"]:
            candidate = result["candidates"][0]
            if "content" in candidate and "parts" in candidate["content"]:
                text = candidate["content"]["parts"][0]["text"]
                _usage = result.get("usageMetadata", {})
                _out_tok = _usage.get("candidatesTokenCount", len(text)//4)
                self._comp.record(success=True, prompt_chars=_prompt_chars,
                                  n_images=_n_imgs, out_tok=_out_tok,
                                  temp=0.1, latency=_latency)
                return text
        
        return "Error: No valid response from Gemini"

    def generate_meta_prompts(self, video_id, crime_type):
        """Meta-prompting phase: Generate optimized prompts for analysis"""
        print("=== META-PROMPTING PHASE: Generating Optimized Analysis Prompts ===")

        meta_prompt = """You are an expert in prompt engineering and crime video analysis. Your task is to design the most effective prompts for analyzing potentially concerning video content.

Given that you will be analyzing video frames that may contain inappropriate or harmful behavior, design specialized prompts for different aspects of the analysis that would maximize accuracy, thoroughness, and reliability.

Create optimized prompts for these 6 key analysis areas:

1. SCENE_ANALYSIS: For understanding the environment and context
2. PERSON_IDENTIFICATION: For identifying individuals and their characteristics
3. BEHAVIOR_ANALYSIS: For analyzing actions and behaviors
4. INTERACTION_DYNAMICS: For understanding relationships and interactions
5. RISK_ASSESSMENT: For identifying concerning elements and red flags
6. COMPREHENSIVE_SYNTHESIS: For creating final assessments

For each area, provide:
- An optimized prompt that would elicit the most accurate and thorough analysis
- Key questions that should be addressed
- Important considerations and potential pitfalls to avoid
- Specific instructions for evidence-based reasoning

Format your response clearly with headers for each analysis area. Make the prompts sophisticated, specific, and designed to maximize analytical quality."""

        contents = [{
            "role": "user",
            "parts": [{"text": meta_prompt}]
        }]

        print("Generating specialized analysis prompts...")
        meta_response = self.make_gemini_request(contents, temperature=0.3)

        # Save meta-prompting phase results
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        meta_result = {
            "meta_prompt_generation": meta_response
        }
        self.save_results(meta_result, f"{crime_type}_{video_id}_meta_generation_{timestamp}.json")

        return meta_response

    def extract_generated_prompts(self, meta_response, video_id, crime_type):
        """Extract and structure the generated prompts"""
        print("Extracting and structuring generated prompts...")

        extraction_prompt = f"""From the following meta-prompting response, extract the specific prompts for each analysis area and format them as a structured JSON object.

Meta-prompting response:
{meta_response}

Extract and format as JSON with this structure:
{{
    "scene_analysis": {{
        "prompt": "extracted prompt text",
        "key_questions": ["question1", "question2", ...],
        "considerations": ["consideration1", "consideration2", ...]
    }},
    "person_identification": {{
        "prompt": "extracted prompt text",
        "key_questions": ["question1", "question2", ...],
        "considerations": ["consideration1", "consideration2", ...]
    }},
    "behavior_analysis": {{
        "prompt": "extracted prompt text",
        "key_questions": ["question1", "question2", ...],
        "considerations": ["consideration1", "consideration2", ...]
    }},
    "interaction_dynamics": {{
        "prompt": "extracted prompt text",
        "key_questions": ["question1", "question2", ...],
        "considerations": ["consideration1", "consideration2", ...]
    }},
    "risk_assessment": {{
        "prompt": "extracted prompt text",
        "key_questions": ["question1", "question2", ...],
        "considerations": ["consideration1", "consideration2", ...]
    }},
    "comprehensive_synthesis": {{
        "prompt": "extracted prompt text",
        "key_questions": ["question1", "question2", ...],
        "considerations": ["consideration1", "consideration2", ...]
    }}
}}

Provide only the JSON structure with the extracted content."""

        contents = [{
            "role": "user",
            "parts": [{"text": extraction_prompt}]
        }]

        extraction_response = self.make_gemini_request(contents, temperature=0.1)

        try:
            # Attempt to parse as JSON, with fallback handling
            import json
            if extraction_response.strip().startswith('{'):
                generated_prompts = json.loads(extraction_response)
            else:
                # If not proper JSON, create a structured fallback
                generated_prompts = self.create_fallback_prompts(meta_response)
        except:
            generated_prompts = self.create_fallback_prompts(meta_response)

        # Save prompt extraction results
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        extraction_result = {
            "prompt_extraction": generated_prompts
        }
        self.save_results(extraction_result, f"{crime_type}_{video_id}_prompt_extraction_{timestamp}.json")

        return generated_prompts

    def create_fallback_prompts(self, meta_response):
        """Create fallback prompts if extraction fails"""
        return {
            "scene_analysis": {
                "prompt": "Analyze the scene environment, setting, location, and overall context visible in these video frames. Consider lighting, objects, spatial layout, and any environmental factors that might be relevant.",
                "key_questions": ["Where is this taking place?", "What type of environment is this?", "What objects and features are visible?"],
                "considerations": ["Note environmental factors that might affect behavior", "Consider privacy vs public settings"]
            },
            "person_identification": {
                "prompt": "Identify all individuals present in the frames. Describe their apparent ages, genders, physical characteristics, clothing, and positioning. Consider their relationships and roles.",
                "key_questions": ["How many people are present?", "What are their apparent ages and characteristics?", "What relationships do they appear to have?"],
                "considerations": ["Be objective about descriptions", "Note power imbalances based on age/size"]
            },
            "behavior_analysis": {
                "prompt": "Analyze the specific behaviors, actions, and movements of each person. Focus on what they are doing, how they are moving, and their physical interactions.",
                "key_questions": ["What specific actions is each person performing?", "How are they moving or positioned?", "What physical interactions are occurring?"],
                "considerations": ["Distinguish between voluntary and involuntary behaviors", "Note any signs of distress or discomfort"]
            },
            "interaction_dynamics": {
                "prompt": "Examine the interpersonal dynamics and relationships between individuals. Analyze communication patterns, body language, and social dynamics.",
                "key_questions": ["How are people interacting with each other?", "What does body language suggest?", "Who appears to be leading interactions?"],
                "considerations": ["Look for signs of consent vs coercion", "Note power dynamics and control patterns"]
            },
            "risk_assessment": {
                "prompt": "Identify any concerning elements, red flags, or indicators of inappropriate or harmful behavior. Focus on signs that warrant concern or further attention.",
                "key_questions": ["What concerning elements are present?", "Are there signs of inappropriate behavior?", "What risks or harms might be indicated?"],
                "considerations": ["Be specific about concerning observations", "Consider context and alternative explanations"]
            },
            "comprehensive_synthesis": {
                "prompt": "Synthesize all previous analyses into a comprehensive assessment. Integrate findings from scene, people, behaviors, interactions, and risks into a coherent conclusion.",
                "key_questions": ["What is the overall picture?", "How do all observations fit together?", "What is the level of concern?"],
                "considerations": ["Base conclusions on evidence", "Acknowledge limitations and uncertainties"]
            }
        }

    def apply_generated_prompts(self, frame_data, generated_prompts, video_id, crime_type):
        """Apply the meta-generated prompts to analyze the frames"""
        print("=== APPLICATION PHASE: Using Generated Prompts for Analysis ===")

        analysis_results = {}

        # Define the analysis sequence
        analysis_sequence = [
            "scene_analysis",
            "person_identification",
            "behavior_analysis",
            "interaction_dynamics",
            "risk_assessment",
            "comprehensive_synthesis"
        ]

        # Apply each generated prompt
        for analysis_type in analysis_sequence:
            if analysis_type in generated_prompts:
                print(f"\nApplying {analysis_type.replace('_', ' ').title()} prompt...")

                prompt_data = generated_prompts[analysis_type]
                optimized_prompt = prompt_data.get("prompt", "")
                key_questions = prompt_data.get("key_questions", [])
                considerations = prompt_data.get("considerations", [])

                # Enhance prompt with key questions and considerations
                full_prompt = f"{optimized_prompt}\n\nKey questions to address:\n"
                for q in key_questions:
                    full_prompt += f"- {q}\n"

                full_prompt += f"\nImportant considerations:\n"
                for c in considerations:
                    full_prompt += f"- {c}\n"

                full_prompt += f"\nProvide a thorough analysis addressing these points."

                # Apply to frame chunks
                type_responses = []
                for i in range(0, len(frame_data), self.chunk_size):
                    chunk = frame_data[i:i + self.chunk_size]

                    _vk_meta = getattr(self, "_vk", "unknown_video")
                    _ck_meta = f"{analysis_type}_chunk_{i//self.chunk_size}"
                    if self._ckpt.is_chunk_done(_vk_meta, _ck_meta):
                        print(f"  [CHECKPOINT] {analysis_type} chunk {i//self.chunk_size+1} loaded from disk")
                        type_responses.append(self._ckpt.get_chunk(_vk_meta, _ck_meta))
                        continue

                    # Prepare content parts for Gemini
                    parts = [{"text": full_prompt}]

                    # Add images to parts
                    for frame in chunk:
                        parts.append({
                            "inline_data": {
                                "mime_type": "image/png",
                                "data": frame
                            }
                        })

                    contents = [{
                        "role": "user",
                        "parts": parts
                    }]

                    response = self.make_gemini_request(contents)
                    type_responses.append(response)
                    if not response.startswith("Error"):
                        self._ckpt.save_chunk(_vk_meta, _ck_meta, response)

                    time.sleep(3)

                # Combine responses for this analysis type
                if len(type_responses) == 1:
                    combined_response = type_responses[0]
                else:
                    combined_response = "\n\n=== NEXT CHUNK ===\n\n".join(type_responses)

                analysis_results[analysis_type] = {
                    "generated_prompt": optimized_prompt,
                    "key_questions": key_questions,
                    "considerations": considerations,
                    "analysis_result": combined_response
                }

                # Save individual analysis type results
                timestamp = time.strftime("%Y%m%d_%H%M%S")
                type_result = {
                    analysis_type: analysis_results[analysis_type]
                }
                self.save_results(type_result, f"{crime_type}_{video_id}_meta_{analysis_type}_{timestamp}.json")

                print(f"Completed {analysis_type.replace('_', ' ')} analysis")

        return analysis_results

    def evaluate_prompt_effectiveness(self, analysis_results, video_id, crime_type):
        """Evaluate the effectiveness of the generated prompts"""
        print("=== EVALUATION PHASE: Assessing Prompt Effectiveness ===")

        evaluation_prompt = f"""Evaluate the effectiveness of the meta-generated prompts based on the analysis results below.

For each analysis type, assess:
1. PROMPT QUALITY: How well did the generated prompt elicit thorough analysis?
2. COMPLETENESS: Did the analysis address all intended aspects?
3. SPECIFICITY: How specific and detailed were the results?
4. RELEVANCE: How relevant were the findings to crime video analysis?
5. ACTIONABILITY: How useful are the insights for decision-making?

Rate each aspect 1-10 and provide improvement suggestions.

Analysis Results:
{json.dumps(analysis_results, indent=2)}

Provide your evaluation in a structured format."""

        contents = [{
            "role": "user",
            "parts": [{"text": evaluation_prompt}]
        }]

        evaluation_response = self.make_gemini_request(contents)

        # Save evaluation results
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        evaluation_result = {
            "effectiveness_evaluation": evaluation_response
        }
        self.save_results(evaluation_result, f"{crime_type}_{video_id}_meta_evaluation_{timestamp}.json")

        return evaluation_response

    def process_frames_meta_prompting(self, frame_data, video_id, crime_type):
        """Process frames using meta-prompting strategy"""
        print("Starting meta-prompting analysis...")

        # Phase 1: Generate optimized prompts
        meta_response = self.generate_meta_prompts(video_id, crime_type)

        # Phase 2: Extract and structure prompts
        generated_prompts = self.extract_generated_prompts(meta_response, video_id, crime_type)

        # Phase 3: Apply generated prompts
        self._vk = f"{crime_type}_{video_id}"
        analysis_results = self.apply_generated_prompts(frame_data, generated_prompts, video_id, crime_type)

        # Phase 4: Evaluate prompt effectiveness
        evaluation = self.evaluate_prompt_effectiveness(analysis_results, video_id, crime_type)

        # Compile complete meta-prompting results
        meta_prompting_results = {
            "meta_prompting_process": {
                "phase_1_meta_generation": meta_response,
                "phase_2_prompt_extraction": generated_prompts,
                "phase_3_analysis_application": analysis_results,
                "phase_4_effectiveness_evaluation": evaluation
            },
            "methodology": {
                "approach": "Meta-prompting with self-generated optimized prompts",
                "phases": [
                    "Meta-prompt generation",
                    "Prompt extraction and structuring",
                    "Application of generated prompts",
                    "Effectiveness evaluation"
                ]
            }
        }

        return meta_prompting_results

    def save_results(self, results, filename):
        """Save results to a file"""
        filepath = os.path.join(self.save_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Results saved to: {filepath}")

    def analyze_frames(self, frames_data, video_id, crime_type):
        """Analyze frames with meta-prompting approach"""
        try:
            frame_names = list(frames_data.keys())

            # Improved sorting function for frame numbers
            def extract_frame_number(filename):
                try:
                    # Handle different naming patterns
                    if '_frame_' in filename:
                        parts = filename.split('_frame_')
                        if len(parts) > 1:
                            number_part = parts[1].split('.')[0]
                            return int(number_part)
                    elif 'frame' in filename.lower():
                        # Alternative pattern matching
                        import re
                        numbers = re.findall(r'\d+', filename)
                        if numbers:
                            return int(numbers[-1])  # Use the last number found
                except Exception as e:
                    print(f"Error extracting frame number from {filename}: {str(e)}")
                    return 0

            sorted_frames = sorted(frame_names, key=extract_frame_number)

            print(f"\n=== ANALYZING VIDEO: {video_id} ({crime_type}) ===")
            print(f"Total frames loaded: {len(frames_data)}")

            _vk = f"{crime_type}_{video_id}"
            if self._ckpt.is_video_complete(_vk):
                print(f"  [CHECKPOINT] {video_id} already complete, loading from disk")
                return self._ckpt.get_video_result(_vk)

            print(f"Frame names sample: {sorted_frames[:5]}{'...' if len(sorted_frames) > 5 else ''}")
            print(f"Using model: {self.model_name}")

            results = {}
            timestamp = time.strftime("%Y%m%d_%H%M%S")

            try:
                frame_data = [frames_data[frame_name] for frame_name in sorted_frames
                             if frame_name in frames_data and frames_data[frame_name]]

                if not frame_data:
                    results["Meta_Prompting_Analysis"] = {
                        "error": "No valid frames were available for analysis.",
                        "frames_used": len(sorted_frames),
                        "valid_frames": 0,
                        "model_used": self.model_name,
                        "crime_type": crime_type,
                        "prompting_technique": "META_PROMPTING"
                    }
                    print("WARNING: No valid frames were available for analysis.")
                else:
                    print(f"Processing {len(frame_data)} valid frames with meta-prompting...")
                    meta_results = self.process_frames_meta_prompting(frame_data, video_id, crime_type)

                    results["Meta_Prompting_Analysis"] = {
                        "method": "meta_prompting",
                        "description": "Self-generated optimized prompts for enhanced analysis",
                        "meta_prompting_results": meta_results,
                        "frames_used": len(sorted_frames),
                        "valid_frames": len(frame_data),
                        "analysis_timestamp": timestamp,
                        "model_used": self.model_name,
                        "crime_type": crime_type,
                        "prompting_technique": "META_PROMPTING"
                    }

                # Save results
                self.save_results(results, f"{crime_type}_{video_id}_meta_prompting_analysis_{timestamp}.json")
                self._ckpt.mark_video_complete(_vk, results)
                self._comp.record_video(len(frame_data))
                print(f"Meta-prompting analysis for {video_id} ({crime_type}) completed and saved.")

            except Exception as e:
                print(f"Error processing meta-prompting analysis: {str(e)}")
                results["Meta_Prompting_Analysis"] = {
                    "method": "meta_prompting",
                    "error": str(e),
                    "frames_used": len(sorted_frames) if 'sorted_frames' in locals() else 0,
                    "model_used": self.model_name,
                    "crime_type": crime_type,
                    "prompting_technique": "META_PROMPTING"
                }

            return results

        except Exception as e:
            print(f"Error in analyze_frames: {str(e)}")
            raise

def discover_all_videos_and_frames(frames_dir):
    """
    Discover pre-extracted frames from FRAMES_DIR.
    Expected layout:
      FRAMES_DIR/<CrimeType>/<VideoID>/frame_000000.png ...
    or (flat):
      FRAMES_DIR/<CrimeType>/<VideoID_frame_XXXXXX>.png ...
    Both layouts are supported automatically.
    """
    print(f"\n=== DISCOVERING PRE-EXTRACTED FRAMES ===")
    print(f"Scanning: {frames_dir}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
    all_videos = {}

    try:
        crime_types = sorted([
            d for d in os.listdir(frames_dir)
            if os.path.isdir(os.path.join(frames_dir, d))
        ])
        print(f"Crime-type folders ({len(crime_types)}): {crime_types}")

        for crime_type in crime_types:
            crime_dir = os.path.join(frames_dir, crime_type)

            # Check for sub-folder layout: <CrimeType>/<VideoID>/<frames>
            sub_dirs = [
                d for d in os.listdir(crime_dir)
                if os.path.isdir(os.path.join(crime_dir, d))
            ]

            if sub_dirs:
                # Sub-folder layout
                for video_id in sorted(sub_dirs):
                    video_frame_dir = os.path.join(crime_dir, video_id)
                    frames = sorted([
                        f for f in os.listdir(video_frame_dir)
                        if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                    ])
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : frames,
                            "crime_dir"  : video_frame_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")
            else:
                # Flat layout: group image files by video_id prefix
                all_files = sorted([
                    f for f in os.listdir(crime_dir)
                    if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                ])
                video_groups = defaultdict(list)
                for fname in all_files:
                    # Extract video_id: everything before _frame_ or last _number
                    base = os.path.splitext(fname)[0]
                    if "_frame_" in base:
                        vid_id = base.split("_frame_")[0]
                    else:
                        vid_id = re.sub(r"_?\d+$", "", base) or base
                    video_groups[vid_id].append(fname)

                for video_id, frames in sorted(video_groups.items()):
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : sorted(frames),
                            "crime_dir"  : crime_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")

    except Exception as e:
        print(f"Error scanning {frames_dir}: {e}")

    print(f"\nTotal videos found: {len(all_videos)}")
    return all_videos
def extract_video_id_from_filename(filename):
    """Legacy stub — video IDs come from folder/file names directly."""
    base = os.path.splitext(filename)[0]
    if "_frame_" in base:
        return base.split("_frame_")[0]
    return re.sub(r"_?\d+$", "", base) or base
def load_frames_for_video(video_info, frame_interval=1):
    """
    Load pre-extracted frame images from disk and base64-encode them.
    Reads from video_info["crime_dir"] which points to the frame folder.
    frame_interval: sample every Nth frame (1 = all frames).
    """
    crime_dir  = video_info["crime_dir"]
    frame_files = video_info["frames"]
    video_id   = video_info["video_id"]

    print(f"\nLoading frames for {video_id} from: {crime_dir}")
    print(f"  Total available: {len(frame_files)} | sampling every {frame_interval}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

    # Sort by embedded frame number
    def _frame_num(fname):
        nums = re.findall(r"\d+", fname)
        return int(nums[-1]) if nums else 0

    sorted_files = sorted(frame_files, key=_frame_num)
    selected     = sorted_files[::frame_interval]
    print(f"  Frames to load: {len(selected)}")

    frames_data = {}
    for idx, fname in enumerate(selected):
        if os.path.splitext(fname.lower())[1] not in IMAGE_EXTS:
            continue
        fpath = os.path.join(crime_dir, fname)
        try:
            with open(fpath, "rb") as fh:
                frames_data[fname] = base64.b64encode(fh.read()).decode("utf-8")
            if idx < 3 or idx % 20 == 0 or idx == len(selected) - 1:
                print(f"  [{idx+1:>5}] {fname} ({os.path.getsize(fpath)/1024:.1f} KB)")
        except Exception as e:
            print(f"  Error loading {fname}: {e}")

    print(f"  Done: {len(frames_data)} frames loaded")
    return frames_data
def process_all_crime_folders():
    """Process ALL crime folders with meta-prompting analysis - PROCESSES ENTIRE DIRECTORY STRUCTURE"""
    # Initialize analyzer
    analyzer = MetaPromptingGeminiAnalyzer()

    # Discover ALL videos and frames in the entire folder structure
    all_videos = discover_all_videos_and_frames(FRAMES_DIR)

    if not all_videos:
        print("No videos found to process!")
        return {}

    all_results = {}
    skipped_videos = []

    print(f"\n🔥 PROCESSING ALL {len(all_videos)} VIDEOS WITH META-PROMPTING 🔥")
    print(f"Using model: {analyzer.model_name}")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("🎯 Meta-Prompting Mode: Self-generated optimized prompts for enhanced analysis!")
    print("📁 ENTIRE FOLDER STRUCTURE WILL BE PROCESSED")
    print("="*70)

    # Process EVERY SINGLE VIDEO discovered
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(video_key, video_info):
        """Process a single video in its own thread.

        Preserves the original per-video checkpoint (analyzer._ckpt) so
        already-completed videos are skipped on resume. Checkpoint reads
        and result-dict writes are protected by checkpoint_lock; stdout
        writes are protected by print_lock.
        """
        with print_lock:
            print(f"\nProcessing video: {video_key}")
            print(f"  Crime type: {video_info['crime_type']}")
            print(f"  Video ID: {video_info['video_id']}")
            print(f"  Frames available: {len(video_info['frames'])}")
        try:
            frames_data = load_frames_for_video(video_info, frame_interval=FRAME_INTERVAL)
            if not frames_data:
                with print_lock:
                    print(f"  No frames loaded for video {video_key} - skipping")
                return video_key, None, "no frames loaded"

            # Resume: skip videos already finished in a prior run
            if analyzer._ckpt.is_video_complete(video_key):
                with print_lock:
                    print(f"  [CHECKPOINT] {video_key} already complete, loading from disk")
                cached = analyzer._ckpt.get_video_result(video_key)
                with checkpoint_lock:
                    all_results[video_key] = cached
                    analyzer._comp.record_video(len(frames_data))
                return video_key, cached, None

            # Run the technique
            results = analyzer.analyze_frames(
                frames_data, video_info['video_id'], video_info['crime_type']
            )
            with checkpoint_lock:
                all_results[video_key] = results
            with print_lock:
                print(f"  Successfully processed {video_key}")
            return video_key, results, None

        except Exception as e:
            with print_lock:
                print(f"  Error processing video {video_key}: {e}")
            return video_key, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(_process_one_video, k, v)
            for k, v in all_videos.items()
        ]
        for fut in as_completed(futures):
            vkey, _res, err = fut.result()
            if err:
                skipped_videos.append(f"{vkey} ({err})")

    # Save summary results
    summary_file = os.path.join(SAVE_DIR, f"meta_prompting_summary_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(summary_file, 'w') as f:
        json.dump(all_results, f, indent=2)

    # Log skipped videos
    if skipped_videos:
        skipped_file = os.path.join(SAVE_DIR, f"skipped_videos_{time.strftime('%Y%m%d_%H%M%S')}.txt")
        with open(skipped_file, 'w') as f:
            f.write("Videos that could not be processed:\n")
            for video in skipped_videos:
                f.write(f"{video}\n")
        print(f"\nSkipped {len(skipped_videos)} videos. List saved to: {skipped_file}")

    print(f"\nComplete meta-prompting analysis saved to: {summary_file}")
    print(f"Successfully processed {len(all_results)} videos out of {len(all_videos)} total")

    return all_results

def test_gemini_api():
    """Test Gemini API connection via Vertex AI genai SDK"""
    print("\nTesting Gemini API connection via Vertex AI...")
    try:
        client = genai.Client(
            vertexai=True,
            project=os.environ["GOOGLE_CLOUD_PROJECT"],
            location="global",
        )
        response = client.models.generate_content(
            model="gemini-3.1-pro-preview",
            contents="Hello, respond with 'API connection successful'"
        )
        if response.text:
            print(f"✓ Gemini API connection successful!")
            print(f"Response: {response.text[:100]}")
            return True
        else:
            print("✗ No response text received")
            return False
    except Exception as e:
        print(f"✗ API connection failed: {e}")
        return False

def check_authentication():
    """Placeholder function to check authentication"""
    return True

def run():
    """Main execution function"""
    print("Meta-Prompting Crime Video Analysis with Gemini - ENTIRE FOLDER PROCESSING")
    print("="*75)
    print("🎯 META-PROMPTING TECHNIQUE: Self-generated optimized prompts for enhanced analysis!")
    print("📁 PROCESSES ALL VIDEOS IN ALL CRIME TYPE FOLDERS")
    print("="*75)

    # Test directory access first
    print("Testing directory access...")
    for path in [DATA_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    # Vertex AI uses GOOGLE_CLOUD_PROJECT env var (no API key file needed)
    print(f"GOOGLE_CLOUD_PROJECT: {os.environ.get('GOOGLE_CLOUD_PROJECT', 'NOT SET')}")

    # Test Gemini API connection
    if not test_gemini_api():
        print("✗ Gemini API test failed. Please check your API key and connection.")
        return

    # Check authentication
    if not check_authentication():
        print("✗ Authentication not completed.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(DATA_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process ALL crime folders - ENTIRE DIRECTORY STRUCTURE
    print("\n🚀 STARTING COMPLETE FOLDER PROCESSING WITH META-PROMPTING...")
    results = process_all_crime_folders()

    # NeurIPS compute report
    if 'analyzer' in dir():
        _report = analyzer._comp.report()
    else:
        _report = {"note": "analyzer not in scope"}
    _rpath = os.path.join(SAVE_DIR, f"neurips_compute_report_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(_rpath, 'w') as _rf:
        json.dump(_report, _rf, indent=2)
    print(f"NeurIPS compute report saved: {_rpath}")

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_phases_completed = 0

    for video_id, video_results in results.items():
        if video_results and 'Meta_Prompting_Analysis' in video_results:
            analysis = video_results['Meta_Prompting_Analysis']
            total_frames_processed += analysis.get('valid_frames', 0)
            if 'meta_prompting_results' in analysis and 'methodology' in analysis['meta_prompting_results']:
                total_phases_completed += len(analysis['meta_prompting_results']['methodology'].get('phases', []))

    print("\n" + "="*75)
    print(f"🎉 COMPLETE META-PROMPTING PROCESSING FINISHED!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total meta-phases completed: {total_phases_completed}")
    print(f"Model used: {analyzer.model_name if 'analyzer' in locals() else 'gemini-3.1-pro-preview'}")
    print(f"Analysis pattern: Generate → Extract → Apply → Evaluate")
    print("📁 ENTIRE FOLDER STRUCTURE WAS PROCESSED")
    print("🎯 Meta-prompting technique applied to all videos")
    print("="*75)

run()

#Chain-Of-Thought Prompting
Chain of Thought (CoT) prompting approach that processes all frames from crime videos. This technique explicitly encourages the model to show its

step-by-step reasoning process.
- Chain of Thought Prompting Approach: The Chain of Thought technique follows this explicit reasoning process:

Step-by-Step Reasoning: The approach explicitly asks the model to "think step by step" through its analysis
- Transparent Reasoning: Each reasoning step is clearly articulated in the response
- Structured Progression: The analysis follows a logical progression from observation to conclusion
- Reasoning Synthesis: The final synthesis also uses step-by-step reasoning to connect all segments

Implementation Highlights

Structured Reasoning Steps:

The prompt breaks down the analysis into 6 clear steps:

- Objective observation without interpretation
- Identification of key actors
- Chronological sequence of events
- Important objects and their usage
- Context and setting analysis
- Integration of observations into a coherent description


Each step builds on the previous one in a logical progression


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full chain of thought process independently


Reasoning-Based Synthesis: The synthesis prompt also follows a chain of thought structure:

- Extraction of key information from each segment
- Timeline construction across all segments
- Tracking people across multiple segments
- Tracking objects across segments
- Contextual integration of segments
- Construction of a comprehensive description


This ensures the synthesis uses the same reasoning approach as individual chunks


Explicit Prompting for Reasoning:

- Both the analysis and synthesis prompts specifically ask to "think step by step"
- System messages reinforce the importance of step-by-step reasoning
The model is explicitly asked to show its thinking process at each step

In [ ]:
import os
import json
import re  
import base64
import requests
from google import genai
from google.genai import types
import time
from datetime import datetime
from collections import defaultdict

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# Running on local machine
# Ensure Google Cloud credentials are set (run cell 5 first)
if "GOOGLE_APPLICATION_CREDENTIALS" not in os.environ:
    print("⚠ WARNING: Run the credentials cell (cell 5) first!")
    print("  GOOGLE_APPLICATION_CREDENTIALS not set")

# Configuration
DATA_DIR   = "C:\\Opeyemi\\PROMPTS\\UCF-Data"   # root with crime-type subfolders
FRAMES_DIR = r"C:\\Opeyemi\\PROMPTS\\FRAMES"  # pre-extracted frames
SAVE_DIR = "C:\\Opeyemi\\PROMPTS\\RESULTS\\GEMINI\\CHAIN-OF-THOUGHT"
FRAME_INTERVAL = 1  # Sample every Nth frame (1 = all frames)
MAX_WORKERS    = 4    # parallel videos processed at once (Gemini/Vertex quotas; raise carefully)
BATCH_SIZE     = 20   # frames per API call (must be defined per-cell so cells run independently)

class ChainOfThoughtGeminiAnalyzer:
    def __init__(self):
        self.model_name = "gemini-3.1-pro-preview"
        self.client = genai.Client(
            vertexai=True,
            project=os.environ["GOOGLE_CLOUD_PROJECT"],
            location="global",
        )
        self.save_dir = SAVE_DIR
        self.chunk_size = BATCH_SIZE
        os.makedirs(self.save_dir, exist_ok=True)
        self._ckpt = CheckpointManager(self.save_dir, type(self).__name__)
        self._comp = ComputeTracker(self.model_name, type(self).__name__)
        # Chain of Thought synthesis prompt
        self.cot_synthesis_prompt = """
You are going to synthesize multiple analyses of different segments of the same video into a coherent understanding of the entire sequence. Use chain of thought reasoning to connect all segments into a complete narrative.

Think through the following steps:

Step 1: Review each segment analysis and extract the key information about people, objects, and actions from each one.

Step 2: Create a timeline by arranging events across all segments in chronological order.

Step 3: Identify which people appear across multiple segments and track their actions throughout.

Step 4: Note how objects or items are used or moved across the entire sequence.

Step 5: Consider the overall context and how different segments relate to each other.

Step 6: Based on all the above reasoning, construct a comprehensive description of what happens throughout the entire video.

Show your thinking at each step as you build your understanding of the complete video sequence.
"""
        # Chain of Thought prompt template
        self.cot_prompt_template = """
Analyze these video frames using a chain of thought reasoning process. Think step by step as you examine what's happening:

Step 1: First, carefully observe and list what you can actually see in the frames. Note people, objects, settings, and actions without interpretation.

Step 2: Identify the key actors in the scene. Describe each person's appearance and what they are doing. Track individuals across multiple frames.

Step 3: Describe the sequence of events chronologically. What happens first, next, and after that?

Step 4: Note any important objects or items in the scene and how they're being used.

Step 5: Consider the context and setting. Where is this taking place? What kind of environment is shown?

Step 6: Based on all the above observations, describe what appears to be happening in these frames.

Make sure to clearly show your thinking process for each step. These are frames {frame_range} of {total_frames}.
"""


    def _call_gemini_sdk_from_payload(self, payload):
        """Convert REST API payload to genai SDK call and return REST-compatible response dict."""
        try:
            # Extract parts from payload
            contents_list = payload.get("contents", [])
            gen_config = payload.get("generationConfig", {})
            
            # Build SDK contents
            sdk_parts = []
            for content_block in contents_list:
                for part in content_block.get("parts", []):
                    if "text" in part:
                        sdk_parts.append(types.Part.from_text(text=part["text"]))
                    elif "inline_data" in part:
                        mime = part["inline_data"].get("mime_type", "image/png")
                        data_bytes = base64.b64decode(part["inline_data"]["data"])
                        sdk_parts.append(types.Part.from_bytes(data=data_bytes, mime_type=mime))
            
            sdk_content = types.Content(role="user", parts=sdk_parts)
            
            # Build generation config
            config = types.GenerateContentConfig(
                temperature=gen_config.get("temperature", 0.1),
                max_output_tokens=gen_config.get("maxOutputTokens", 4096),
                top_p=gen_config.get("topP", 0.8),
                top_k=gen_config.get("topK", 10),
            )
            
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=sdk_content,
                config=config,
            )
            
            # Convert to REST-compatible dict
            if response.text:
                return {
                    "candidates": [{
                        "content": {
                            "parts": [{"text": response.text}]
                        }
                    }],
                    "usageMetadata": {
                        "promptTokenCount": getattr(response.usage_metadata, 'prompt_token_count', 0) if response.usage_metadata else 0,
                        "candidatesTokenCount": getattr(response.usage_metadata, 'candidates_token_count', 0) if response.usage_metadata else 0,
                    }
                }
            else:
                return {"error": "No text in response", "candidates": []}
                
        except Exception as e:
            error_msg = str(e)
            print(f"API Error: {error_msg}")
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "quota" in error_msg.lower():
                print("Rate limited - waiting 60 seconds...")
                time.sleep(60)
                return self._call_gemini_sdk_from_payload(payload)  # Retry once
            return {"error": error_msg}



    def make_gemini_request(self, contents, temperature=0.1):
        """Make a single request to Gemini API"""
        payload = {
            "contents": contents,
            "generationConfig": {
                "temperature": 0.1,
                "maxOutputTokens": 4096,
                "topP": 0.8,
                "topK": 10
            }
        }
        _t0_req = time.time()
        result = self._call_gemini_sdk_from_payload(payload)
        _latency = time.time() - _t0_req
        _prompt_chars = len(str(payload))
        _n_imgs = sum(1 for c in contents for p in c.get("parts", []) if "inline_data" in p)
        
        if isinstance(result, dict) and "error" in result:
            print(f"API Error: {result['error']}")
            self._comp.record(success=False, prompt_chars=_prompt_chars,
                              n_images=_n_imgs, out_tok=0,
                              temp=0.1, latency=_latency)
            return f"Error: {result['error']}"
        
        if "candidates" in result and result["candidates"]:
            candidate = result["candidates"][0]
            if "content" in candidate and "parts" in candidate["content"]:
                text = candidate["content"]["parts"][0]["text"]
                _usage = result.get("usageMetadata", {})
                _out_tok = _usage.get("candidatesTokenCount", len(text)//4)
                self._comp.record(success=True, prompt_chars=_prompt_chars,
                                  n_images=_n_imgs, out_tok=_out_tok,
                                  temp=0.1, latency=_latency)
                return text
        
        return "Error: No valid response from Gemini"

    def _format_chunk_analyses(self, all_chunk_analyses):
        """Helper method to format chunk analyses for synthesis"""
        formatted_chunks = []
        for analysis in all_chunk_analyses:
            chunk_text = f"SEGMENT {analysis['chunk']} (Frames {analysis['frame_range']}):\n{analysis['analysis']}\n\n{'-' * 40}\n"
            formatted_chunks.append(chunk_text)

        return ''.join(formatted_chunks)

    def process_frames_with_cot(self, frames_data, video_id, crime_type):
        """Process frames with chain of thought prompting approach"""
        # Extract frame data from the dictionary
        frame_names = list(frames_data.keys())

        # Improved sorting function for frame numbers
        def extract_frame_number(filename):
            try:
                # Handle different naming patterns
                if '_frame_' in filename:
                    parts = filename.split('_frame_')
                    if len(parts) > 1:
                        number_part = parts[1].split('.')[0]
                        return int(number_part)
                elif 'frame' in filename.lower():
                    # Alternative pattern matching
                    import re
                    numbers = re.findall(r'\d+', filename)
                    if numbers:
                        return int(numbers[-1])  # Use the last number found
            except Exception as e:
                print(f"Error extracting frame number from {filename}: {str(e)}")
                return 0

        sorted_frames = sorted(frame_names, key=extract_frame_number)
        frame_data = [frames_data[frame_name] for frame_name in sorted_frames if frame_name in frames_data and frames_data[frame_name]]

        if not frame_data:
            return {"error": "No valid frames available for analysis"}

        total_frames = len(frame_data)
        print(f"Processing all {total_frames} frames with chain of thought approach")

        # Process all frames by dividing them into chunks
        chunk_size = BATCH_SIZE
        frame_chunks = [frame_data[i:i+chunk_size] for i in range(0, total_frames, chunk_size)]
        print(f"Split into {len(frame_chunks)} chunks of approximately {chunk_size} frames each")

        # Initialize results
        cot_results = {}
        all_chunk_analyses = []

        # Process each chunk of frames
        for chunk_idx, chunk in enumerate(frame_chunks):
            frame_start = chunk_idx * chunk_size + 1
            frame_end = min((chunk_idx + 1) * chunk_size, total_frames)
            frame_range = f"{frame_start}-{frame_end}"

            print(f"Processing chunk {chunk_idx+1}/{len(frame_chunks)} (frames {frame_range})...")

            _vk_cot = f"{crime_type}_{video_id}"
            _ck = f"chunk_{chunk_idx}"
            if self._ckpt.is_chunk_done(_vk_cot, _ck):
                print(f"  [CHECKPOINT] CoT chunk {chunk_idx+1} already done")
                _cached = self._ckpt.get_chunk(_vk_cot, _ck)
                chunk_results["cot_analysis"] = _cached
                all_chunk_analyses.append({"chunk": chunk_idx+1, "frame_range": frame_range, "analysis": _cached})
                cot_results[f"Chunk {chunk_idx+1}"] = chunk_results
                continue

            # Initialize chunk results
            chunk_results = {
                "frame_range": frame_range
            }

            # Format the CoT prompt for this chunk
            formatted_cot_prompt = self.cot_prompt_template.format(frame_range=frame_range, total_frames=total_frames)

            # Prepare content parts for Gemini
            parts = [{"text": formatted_cot_prompt}]

            # Add frames to parts
            for frame in chunk:
                parts.append({
                    "inline_data": {
                        "mime_type": "image/png",
                        "data": frame
                    }
                })

            # Create contents for Gemini
            contents = [{
                "role": "user",
                "parts": parts
            }]

            try:
                print(f"  Sending request to Gemini for chunk {chunk_idx+1}...")
                cot_analysis = self.make_gemini_request(contents, temperature=0.1)

                if not cot_analysis.startswith("Error"):
                    print(f"  Received response for chunk {chunk_idx+1}")

                    # Save the CoT analysis
                    chunk_results["cot_analysis"] = cot_analysis
                    self._ckpt.save_chunk(_vk_cot, _ck, cot_analysis)

                    # Add to collection of all chunk analyses
                    all_chunk_analyses.append({
                        "chunk": chunk_idx + 1,
                        "frame_range": frame_range,
                        "analysis": cot_analysis
                    })
                else:
                    print(f"  Error in CoT analysis for chunk {chunk_idx+1}: {cot_analysis}")
                    chunk_results["error"] = cot_analysis

            except Exception as e:
                print(f"  Error in CoT analysis for chunk {chunk_idx+1}: {str(e)}")
                chunk_results["error"] = str(e)

            # Save results for this chunk
            cot_results[f"Chunk {chunk_idx+1}"] = chunk_results

            # Save intermediate results for this chunk
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            chunk_result = {
                f"Chunk {chunk_idx+1}": chunk_results
            }
            self.save_results(chunk_result, f"{crime_type}_{video_id}_cot_chunk{chunk_idx+1}_{timestamp}.json")
            print(f"  Results for chunk {chunk_idx+1} saved")

            # Rate limiting between chunks
            print(f"  Waiting 3 seconds before next chunk...")
            time.sleep(3)

        # After processing all chunks, generate a synthesis using CoT
        if all_chunk_analyses:
            print("Generating chain of thought synthesis across all chunks...")

            # Create synthesis prompt with all chunk analyses
            synthesis_text = f"""
{self.cot_synthesis_prompt}

Here are the analyses for each segment of the video:

{'-' * 40}
{self._format_chunk_analyses(all_chunk_analyses)}

Think step by step to synthesize these segments into a complete understanding of the video.
"""

            # Create contents for synthesis
            synthesis_contents = [{
                "role": "user",
                "parts": [{"text": synthesis_text}]
            }]

            try:
                print("Sending request for final synthesis...")
                cot_synthesis = self.make_gemini_request(synthesis_contents, temperature=0.1)

                if not cot_synthesis.startswith("Error"):
                    print("Synthesis complete!")

                    # Save CoT synthesis
                    cot_results["Chain of Thought Synthesis"] = {
                        "synthesis": cot_synthesis
                    }

                    # Save synthesis separately
                    timestamp = time.strftime("%Y%m%d_%H%M%S")
                    synthesis_result = {
                        "Chain of Thought Synthesis": cot_results["Chain of Thought Synthesis"]
                    }
                    self.save_results(synthesis_result, f"{crime_type}_{video_id}_cot_synthesis_{timestamp}.json")
                    print("Synthesis results saved")
                else:
                    print(f"Error in CoT synthesis: {cot_synthesis}")
                    cot_results["Chain of Thought Synthesis"] = {
                        "error": cot_synthesis
                    }
            except Exception as e:
                print(f"Error in CoT synthesis: {str(e)}")
                cot_results["Chain of Thought Synthesis"] = {
                    "error": str(e)
                }

        return {
            "cot_results": cot_results,
            "frames_used": total_frames,
            "chunks_processed": len(frame_chunks),
            "frames_per_chunk": chunk_size,
            "model_used": self.model_name,  # Track which model was used
            "crime_type": crime_type,
            "prompting_technique": "CHAIN_OF_THOUGHT_PROMPTING"
        }

    def save_results(self, results, filename):
        """Save results to a file"""
        filepath = os.path.join(self.save_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Results saved to: {filepath}")

    def analyze_frames(self, frames_data, video_id, crime_type):
        """Analyze frames with chain of thought prompting"""
        try:
            print(f"\n=== ANALYZING VIDEO: {video_id} ({crime_type}) WITH CHAIN OF THOUGHT PROMPTING ===")
            print(f"Total frames loaded: {len(frames_data)}")
            print(f"Using model: {self.model_name}")

            _vk = f"{crime_type}_{video_id}"
            if self._ckpt.is_video_complete(_vk):
                print(f"  [CHECKPOINT] {video_id} already complete, loading from disk")
                return self._ckpt.get_video_result(_vk)

            timestamp = time.strftime("%Y%m%d_%H%M%S")
            results = self.process_frames_with_cot(frames_data, video_id, crime_type)

            # Save complete results
            self.save_results(results, f"{crime_type}_{video_id}_cot_complete_{timestamp}.json")
            self._ckpt.mark_video_complete(_vk, results)
            self._comp.record_video(len(frames_data))
            print(f"Complete chain of thought analysis for {video_id} ({crime_type}) saved.")

            return results

        except Exception as e:
            print(f"Error in chain of thought analysis: {str(e)}")
            return {"error": str(e)}

def discover_all_videos_and_frames(frames_dir):
    """
    Discover pre-extracted frames from FRAMES_DIR.
    Expected layout:
      FRAMES_DIR/<CrimeType>/<VideoID>/frame_000000.png ...
    or (flat):
      FRAMES_DIR/<CrimeType>/<VideoID_frame_XXXXXX>.png ...
    Both layouts are supported automatically.
    """
    print(f"\n=== DISCOVERING PRE-EXTRACTED FRAMES ===")
    print(f"Scanning: {frames_dir}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
    all_videos = {}

    try:
        crime_types = sorted([
            d for d in os.listdir(frames_dir)
            if os.path.isdir(os.path.join(frames_dir, d))
        ])
        print(f"Crime-type folders ({len(crime_types)}): {crime_types}")

        for crime_type in crime_types:
            crime_dir = os.path.join(frames_dir, crime_type)

            # Check for sub-folder layout: <CrimeType>/<VideoID>/<frames>
            sub_dirs = [
                d for d in os.listdir(crime_dir)
                if os.path.isdir(os.path.join(crime_dir, d))
            ]

            if sub_dirs:
                # Sub-folder layout
                for video_id in sorted(sub_dirs):
                    video_frame_dir = os.path.join(crime_dir, video_id)
                    frames = sorted([
                        f for f in os.listdir(video_frame_dir)
                        if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                    ])
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : frames,
                            "crime_dir"  : video_frame_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")
            else:
                # Flat layout: group image files by video_id prefix
                all_files = sorted([
                    f for f in os.listdir(crime_dir)
                    if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                ])
                video_groups = defaultdict(list)
                for fname in all_files:
                    # Extract video_id: everything before _frame_ or last _number
                    base = os.path.splitext(fname)[0]
                    if "_frame_" in base:
                        vid_id = base.split("_frame_")[0]
                    else:
                        vid_id = re.sub(r"_?\d+$", "", base) or base
                    video_groups[vid_id].append(fname)

                for video_id, frames in sorted(video_groups.items()):
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : sorted(frames),
                            "crime_dir"  : crime_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")

    except Exception as e:
        print(f"Error scanning {frames_dir}: {e}")

    print(f"\nTotal videos found: {len(all_videos)}")
    return all_videos
def extract_video_id_from_filename(filename):
    """Legacy stub — video IDs come from folder/file names directly."""
    base = os.path.splitext(filename)[0]
    if "_frame_" in base:
        return base.split("_frame_")[0]
    return re.sub(r"_?\d+$", "", base) or base
def load_frames_for_video(video_info, frame_interval=1):
    """
    Load pre-extracted frame images from disk and base64-encode them.
    Reads from video_info["crime_dir"] which points to the frame folder.
    frame_interval: sample every Nth frame (1 = all frames).
    """
    crime_dir  = video_info["crime_dir"]
    frame_files = video_info["frames"]
    video_id   = video_info["video_id"]

    print(f"\nLoading frames for {video_id} from: {crime_dir}")
    print(f"  Total available: {len(frame_files)} | sampling every {frame_interval}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

    # Sort by embedded frame number
    def _frame_num(fname):
        nums = re.findall(r"\d+", fname)
        return int(nums[-1]) if nums else 0

    sorted_files = sorted(frame_files, key=_frame_num)
    selected     = sorted_files[::frame_interval]
    print(f"  Frames to load: {len(selected)}")

    frames_data = {}
    for idx, fname in enumerate(selected):
        if os.path.splitext(fname.lower())[1] not in IMAGE_EXTS:
            continue
        fpath = os.path.join(crime_dir, fname)
        try:
            with open(fpath, "rb") as fh:
                frames_data[fname] = base64.b64encode(fh.read()).decode("utf-8")
            if idx < 3 or idx % 20 == 0 or idx == len(selected) - 1:
                print(f"  [{idx+1:>5}] {fname} ({os.path.getsize(fpath)/1024:.1f} KB)")
        except Exception as e:
            print(f"  Error loading {fname}: {e}")

    print(f"  Done: {len(frames_data)} frames loaded")
    return frames_data
def process_all_crime_folders():
    """Process ALL crime folders with chain of thought prompting - PROCESSES ENTIRE DIRECTORY STRUCTURE"""
    # Initialize analyzer
    analyzer = ChainOfThoughtGeminiAnalyzer()

    # Discover ALL videos and frames in the entire folder structure
    all_videos = discover_all_videos_and_frames(FRAMES_DIR)

    if not all_videos:
        print("No videos found to process!")
        return {}

    all_results = {}
    skipped_videos = []

    print(f"\n🔥 PROCESSING ALL {len(all_videos)} VIDEOS WITH CHAIN OF THOUGHT PROMPTING 🔥")
    print(f"Using model: {analyzer.model_name}")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("🎯 Chain of Thought Mode: Step-by-step reasoning with explicit thinking process!")
    print("📁 ENTIRE FOLDER STRUCTURE WILL BE PROCESSED")
    print("="*70)

    # Process EVERY SINGLE VIDEO discovered
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(video_key, video_info):
        """Process a single video in its own thread.

        Preserves the original per-video checkpoint (analyzer._ckpt) so
        already-completed videos are skipped on resume. Checkpoint reads
        and result-dict writes are protected by checkpoint_lock; stdout
        writes are protected by print_lock.
        """
        with print_lock:
            print(f"\nProcessing video: {video_key}")
            print(f"  Crime type: {video_info['crime_type']}")
            print(f"  Video ID: {video_info['video_id']}")
            print(f"  Frames available: {len(video_info['frames'])}")
        try:
            frames_data = load_frames_for_video(video_info, frame_interval=FRAME_INTERVAL)
            if not frames_data:
                with print_lock:
                    print(f"  No frames loaded for video {video_key} - skipping")
                return video_key, None, "no frames loaded"

            # Resume: skip videos already finished in a prior run
            if analyzer._ckpt.is_video_complete(video_key):
                with print_lock:
                    print(f"  [CHECKPOINT] {video_key} already complete, loading from disk")
                cached = analyzer._ckpt.get_video_result(video_key)
                with checkpoint_lock:
                    all_results[video_key] = cached
                    analyzer._comp.record_video(len(frames_data))
                return video_key, cached, None

            # Run the technique
            results = analyzer.analyze_frames(
                frames_data, video_info['video_id'], video_info['crime_type']
            )
            with checkpoint_lock:
                all_results[video_key] = results
            with print_lock:
                print(f"  Successfully processed {video_key}")
            return video_key, results, None

        except Exception as e:
            with print_lock:
                print(f"  Error processing video {video_key}: {e}")
            return video_key, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(_process_one_video, k, v)
            for k, v in all_videos.items()
        ]
        for fut in as_completed(futures):
            vkey, _res, err = fut.result()
            if err:
                skipped_videos.append(f"{vkey} ({err})")

    # Save summary results
    summary_file = os.path.join(SAVE_DIR, f"cot_summary_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(summary_file, 'w') as f:
        json.dump(all_results, f, indent=2)

    # Log skipped videos
    if skipped_videos:
        skipped_file = os.path.join(SAVE_DIR, f"skipped_videos_{time.strftime('%Y%m%d_%H%M%S')}.txt")
        with open(skipped_file, 'w') as f:
            f.write("Videos that could not be processed:\n")
            for video in skipped_videos:
                f.write(f"{video}\n")
        print(f"\nSkipped {len(skipped_videos)} videos. List saved to: {skipped_file}")

    print(f"\nComplete chain of thought analysis saved to: {summary_file}")
    print(f"Successfully processed {len(all_results)} videos out of {len(all_videos)} total")

    return all_results

def test_gemini_api():
    """Test Gemini API connection via Vertex AI genai SDK"""
    print("\nTesting Gemini API connection via Vertex AI...")
    try:
        client = genai.Client(
            vertexai=True,
            project=os.environ["GOOGLE_CLOUD_PROJECT"],
            location="global",
        )
        response = client.models.generate_content(
            model="gemini-3.1-pro-preview",
            contents="Hello, respond with 'API connection successful'"
        )
        if response.text:
            print(f"✓ Gemini API connection successful!")
            print(f"Response: {response.text[:100]}")
            return True
        else:
            print("✗ No response text received")
            return False
    except Exception as e:
        print(f"✗ API connection failed: {e}")
        return False

def run():
    """Main execution function"""
    print("Chain of Thought Prompting Crime Video Analysis with Gemini - ENTIRE FOLDER PROCESSING")
    print("="*75)
    print("🎯 CHAIN OF THOUGHT TECHNIQUE: Step-by-step reasoning with explicit thinking process!")
    print("📁 PROCESSES ALL VIDEOS IN ALL CRIME TYPE FOLDERS")
    print("="*75)

    # Test directory access first
    print("Testing directory access...")
    for path in [DATA_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    # Vertex AI uses GOOGLE_CLOUD_PROJECT env var (no API key file needed)
    print(f"GOOGLE_CLOUD_PROJECT: {os.environ.get('GOOGLE_CLOUD_PROJECT', 'NOT SET')}")

    # Test Gemini API connection
    if not test_gemini_api():
        print("✗ Gemini API test failed. Please check your API key and connection.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(DATA_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process ALL crime folders - ENTIRE DIRECTORY STRUCTURE
    print("\n🚀 STARTING COMPLETE FOLDER PROCESSING WITH CHAIN OF THOUGHT PROMPTING...")
    results = process_all_crime_folders()

    # NeurIPS compute report
    if 'analyzer' in dir():
        _report = analyzer._comp.report()
    else:
        _report = {"note": "analyzer not in scope"}
    _rpath = os.path.join(SAVE_DIR, f"neurips_compute_report_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(_rpath, 'w') as _rf:
        json.dump(_report, _rf, indent=2)
    print(f"NeurIPS compute report saved: {_rpath}")

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_chunks_processed = 0

    for video_id, video_results in results.items():
        if video_results and 'cot_results' in video_results:
            total_frames_processed += video_results.get('frames_used', 0)
            total_chunks_processed += video_results.get('chunks_processed', 0)

    print("\n" + "="*75)
    print(f"🎉 COMPLETE CHAIN OF THOUGHT PROCESSING FINISHED!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total chunks processed: {total_chunks_processed}")
    print(f"Model used: {analyzer.model_name if 'analyzer' in locals() else 'gemini-3.1-pro-preview'}")
    print(f"Analysis pattern: Step 1 → Step 2 → ... → Step 6 → Synthesis")
    print("📁 ENTIRE FOLDER STRUCTURE WAS PROCESSED")
    print("🎯 Chain of thought prompting technique applied to all videos")
    print("="*75)

run()